In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage, AIMessage,BaseMessage
from langchain_core.chat_history import BaseChatMessageHistory
from pydantic import BaseModel, Field
from typing import List
load_dotenv()

True

In [9]:
class WindowChatMessageHistory(BaseChatMessageHistory,BaseModel):
    messages: List[BaseMessage] = Field(default_factory=list)
    k : int = Field(default=6, description="The number of messages to keep in the sliding window.")
    def add_messages(self, messages):
        self.messages.extend(messages)
        # k = 6 means 3 human message. + 3 AI messages = 3 turn
        if len(self.messages) > self.k:
            dropped = len(self.messages) - self.k
            self.messages = self.messages[-self.k:]

    def clear(self):
        self.messages = []



In [10]:
model = ChatOpenAI(model_name="gpt-4o-mini")


In [11]:
WINDOW_K = 8
prompt = ChatPromptTemplate.from_messages([
    ("system", """\
You are a professional email support agent.
Classify: Billing / Technical / General. Priority: High / Medium / Low.
Remember customer details within the conversation."""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chain = prompt | model | StrOutputParser()

In [12]:
window_store = {}

def get_session_history(session_id):
    if session_id not in window_store:
        window_store[session_id] = WindowChatMessageHistory(k=WINDOW_K)
    return window_store[session_id]

In [13]:
window_assistant = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

/Users/rahultiwari/Documents/wills_18th_july_batch/ai-env/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [25]:
cfg = {"configurable": {"session_id": "window_john"}}

r1 = window_assistant.invoke({"input":"Draft a 2-line apology email for him."},
                        config=cfg)

In [26]:
r1

'Subject: Apology Regarding Invoice Discrepancy\n\nDear John,  \n\nWe sincerely apologize for the overcharge on invoice #1042 and appreciate your patience as we resolve this issue promptly. Please rest assured that we are addressing it and will follow up shortly.  \n\nBest regards,  \n[Your Name]  \n[Your Position]  '

In [27]:
window_store['window_john'].messages

[HumanMessage(content='What category is his complaint?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="The category of John's complaint is **Billing**.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What is the SLA for billing?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="The standard Service Level Agreement (SLA) for billing inquiries typically ranges from 24 to 48 hours for a response, depending on the company's policy. However, specific SLAs may vary, so it's important to refer to your organization's guidelines for precise information. If John's issue is classified as high priority, it may be addressed even more urgently.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Can you remind me who made the original complaint?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The original complaint was made by **Jo

In [6]:
memory = WindowChatMessageHistory()

In [ ]:
message = ['m3', 'm4', 'm5', 'm6','m7']

In [8]:
message[1:]

['m2', 'm3', 'm4', 'm5', 'm6']